# Medical appointment data: exploratory data analysis\n\nThis EDA uses the original CSV. It documents quality, distributions, missingness, appointment outcomes, and calendar trends.\n

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from clinic_ml import load_analysis_data
Path('artifacts').mkdir(exist_ok=True)
df = load_analysis_data(Path('Medical_appointment_data.csv'))
print(df.shape)
display(df.head())


(109593, 36)


,specialty,appointment_time,gender,no_show,disability,place,appointment_shift,age,under_12_years_old,over_60_years_old,patient_needs_companion,average_temp_day,average_rain_day,max_temp_day,max_rain_day,rainy_day_before,storm_day_before,rain_intensity,heat_intensity,appointment_date_continuous,hipertension,diabetes,alcoholism,handcap,scholarship,sms_received,appointment_hour,under_12,over_60,day_of_week,month_sin,month_cos,day_of_year_sin,day_of_year_cos,companion_child,chronic_count
0,psychotherapy,17,F,yes,intellectual,Lake Marvinville,afternoon,9.0,1,0,1,23.18,0.00,27.5,0.0,1,1,no_rain,warm,2020-01-01,0,0,0,0,0,0,17,1.0,0.0,2,0.5,0.866025,0.017202,0.999852,1.0,0
1,Unknown,7,M,no,intellectual,ITAPEMA,morning,11.0,1,0,1,14.31,0.02,16.5,0.6,1,1,no_rain,cold,2020-01-01,0,0,0,0,0,0,7,1.0,0.0,2,0.5,0.866025,0.017202,0.999852,1.0,0
2,speech therapy,16,M,no,intellectual,ITAJAÍ,afternoon,8.0,1,0,1,21.61,0.01,29.9,0.2,1,1,no_rain,warm,2020-01-01,0,0,0,0,0,0,16,1.0,0.0,2,0.5,0.866025,0.017202,0.999852,1.0,0
3,speech therapy,14,M,yes,intellectual,Sarahside,afternoon,9.0,1,0,1,21.39,0.11,24.1,1.4,1,1,moderate,mild,2020-01-01,0,0,0,0,0,1,14,1.0,0.0,2,0.5,0.866025,0.017202,0.999852,1.0,0
4,physiotherapy,8,M,no,motor,ITAJAÍ,morning,NaN,0,0,0,20.15,0.02,23.1,0.2,1,1,no_rain,mild,2020-01-01,0,0,0,0,0,0,8,NaN,NaN,2,0.5,0.866025,0.017202,0.999852,NaN,0


## Missing-value handling

The specified source columns are excluded before analysis. Numeric values are filled with the mean when their distribution is approximately symmetric and the median when it is skewed; other categorical values use the mode. `specialty`, `disability`, and the outcome are not imputed.


In [ ]:
# Exclude the requested source columns from this notebook's analysis only.
excluded_columns = [
    'sms_received', 'scholarship', 'under_12_years_old', 'over_60_years_old',
    'hipertension', 'alcoholism', 'handcap',
]
df = df.drop(columns=excluded_columns, errors='ignore')

# Keep these fields unchanged: specialty and disability may be meaningfully missing,
# and no_show is the outcome label rather than a predictor to impute.
protected_columns = {'specialty', 'disability', 'no_show'}
fill_summary = []

for column in df.select_dtypes(include='number').columns:
    if column in protected_columns or not df[column].isna().any():
        continue
    values = df[column].dropna()
    if values.empty:
        continue
    if abs(values.skew()) <= 1:
        method, fill_value = 'mean', values.mean()
    else:
        method, fill_value = 'median', values.median()
    missing_count = int(df[column].isna().sum())
    df[column] = df[column].fillna(fill_value)
    fill_summary.append({'column': column, 'method': method, 'values_filled': missing_count})

for column in df.select_dtypes(include=['object', 'string', 'category']).columns:
    if column in protected_columns or not df[column].isna().any():
        continue
    mode = df[column].mode(dropna=True)
    if mode.empty:
        continue
    missing_count = int(df[column].isna().sum())
    df[column] = df[column].fillna(mode.iloc[0])
    fill_summary.append({'column': column, 'method': 'mode', 'values_filled': missing_count})

fill_summary = pd.DataFrame(fill_summary, columns=['column', 'method', 'values_filled'])
print('Excluded columns:', ', '.join(excluded_columns))
if fill_summary.empty:
    print('No eligible missing values required imputation.')
else:
    display(fill_summary)


## One-hot encoding

Categorical predictors are converted to binary indicator columns. The target is not encoded.


In [ ]:
# One-hot encode categorical predictors and save the model-ready dataset.
# Place is deliberately excluded: it has thousands of distinct location values, which makes one-hot encoding impractically wide.
categorical_features = ['gender', 'specialty', 'disability', 'appointment_shift', 'age_group', 'appointment_time_band', 'appointment_day_of_week', 'rain_intensity', 'heat_intensity']
categorical_features = [column for column in categorical_features if column in df.columns]
from time import perf_counter

start = perf_counter()
X = df.drop(columns=['no_show'])
encoded_data = pd.get_dummies(X, columns=categorical_features, dummy_na=True, dtype='int8')
encoded_data['no_show'] = df['no_show'].map({'no': 0, 'yes': 1})

# Save the encoded matrix as a separate exploratory artifact.
Path('artifacts').mkdir(exist_ok=True)
output_path = 'artifacts/notebook_encoded.csv'
encoded_data.to_csv(output_path, index=False)

print('Categorical features encoded:', categorical_features)
print(f'Encoded dataset: {encoded_data.shape[0]:,} rows x {encoded_data.shape[1]:,} columns')
print(f'Saved one-hot encoded data to: {output_path}')
print(f'Completed in {perf_counter() - start:.2f} seconds')
display(encoded_data.head())

# Review the fields saved to artifacts/notebook_encoded.csv.
# For numeric fields, the mean is also useful: for a one-hot field it is the
# proportion of appointments represented by that category.
numeric_means = encoded_data.mean(numeric_only=True).reindex(encoded_data.columns).round(4)
encoded_field_summary = pd.DataFrame({
    'field': encoded_data.columns,
    'data_type': encoded_data.dtypes.astype(str).to_numpy(),
    'example_values': [', '.join(map(str, encoded_data[column].dropna().unique()[:3]))
                       for column in encoded_data.columns],
    'mean': numeric_means.to_numpy(),
})
display(encoded_field_summary)


In [ ]:
quality = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean()*100).round(2),
    'unique_values': df.nunique(dropna=True)
}).sort_values('missing_pct', ascending=False)
display(df.dtypes.to_frame('dtype').T)
display(quality)
print('Exact duplicate rows:', df.duplicated().sum())
dates = df['appointment_date_continuous'].dropna()
print('Calendar range:', dates.min().date(), 'to', dates.max().date())
print('Missing calendar days:', len(pd.date_range(dates.min(), dates.max()).difference(dates.unique())))


In [ ]:
target = df['no_show'].value_counts().rename_axis('no_show').reset_index(name='appointments')
target['pct'] = (100 * target['appointments'] / len(df)).round(2)
display(target)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x='no_show', order=['no','yes'], ax=axes[0], palette=['#4C78A8','#E45756'])
axes[0].set(title='Appointment outcome distribution', xlabel='No show', ylabel='Appointments')
sns.histplot(data=df, x='age', bins=35, ax=axes[1], color='#4C78A8')
axes[1].set(title='Age distribution', xlabel='Age')
plt.tight_layout(); plt.show()


In [ ]:
missing = quality.query('missing_count > 0').sort_values('missing_pct')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(missing.index, missing['missing_pct'], color='#F2CF5B')
ax.set(title='Missing values by field', xlabel='Missing (%)', ylabel='')
for i, value in enumerate(missing['missing_pct']):
    ax.text(value + .15, i, f'{value:.1f}%')
plt.show()

def no_show_by(column, top_n=12):
    from clinic_ml import no_show_summary
    return no_show_summary(df, column, top_n)


for field in [
    'gender', 'appointment_shift', 'specialty', 'disability',
    'place', 'under_12', 'over_60', 'patient_needs_companion'
]:
    display(no_show_by(field))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, field, title in zip(axes, ['specialty','place'], ['No-show rate by specialty', 'No-show rate by city (top 12 volume)']):
    rates = no_show_by(field).sort_values('no_show_rate')
    sns.barplot(data=rates, x='no_show_rate', y=field, ax=ax, color='#E45756')
    ax.set(title=title, xlabel='No-show rate (%)', ylabel='')
plt.tight_layout(); plt.show()

daily = (df.assign(no_show_flag=df['no_show'].eq('yes')).groupby('appointment_date_continuous')
         .agg(appointments=('no_show','size'), no_show_rate=('no_show_flag','mean')).reset_index())
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
sns.lineplot(data=daily, x='appointment_date_continuous', y='appointments', ax=axes[0], color='#4C78A8')
sns.lineplot(data=daily, x='appointment_date_continuous', y='no_show_rate', ax=axes[1], color='#E45756')
axes[0].set(title='Daily appointment volume', xlabel='', ylabel='Appointments')
axes[1].set(title='Daily no-show rate', xlabel='Appointment date', ylabel='Rate')
axes[1].yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
plt.tight_layout(); plt.show()


## Outlier detection

Potential outliers are identified with the interquartile range (IQR) rule. This is a diagnostic only and does not remove or modify records.


In [ ]:
# IQR outlier detection for continuous numeric fields.
outlier_columns = [column for column in ['age', 'appointment_time', 'avg_temp', 'average_temp_day', 'max_temp', 'max_temp_day', 'avg_rain', 'average_rain_day', 'max_rain', 'max_rain_day'] if column in df.columns]
outlier_summary = []
for column in outlier_columns:
    values = pd.to_numeric(df[column], errors='coerce').dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((values < lower_bound) | (values > upper_bound)).sum())
    outlier_summary.append({'feature': column, 'lower_bound': lower_bound, 'upper_bound': upper_bound, 'outlier_count': outlier_count, 'outlier_pct': outlier_count / len(values) * 100})

outlier_summary = pd.DataFrame(outlier_summary).sort_values('outlier_count', ascending=False)
display(outlier_summary.round(2))

fig, axes = plt.subplots(1, len(outlier_columns), figsize=(4 * len(outlier_columns), 4))
axes = [axes] if len(outlier_columns) == 1 else axes
for ax, column in zip(axes, outlier_columns):
    sns.boxplot(y=df[column], ax=ax, color='#72B7B2')
    ax.set_title(column)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()


## Interpretation note\n\nThe charts are descriptive, not causal. Preserve missingness in later modeling and ensure all predictor values are known before the appointment.\n